# This python notebook is designed to teach and introduce basic concepts of supervised machine learning for geoscience. More specifically, we will use random forest to examine the relationship of total organic carbon (TOC) and X-ray fluoresence-derived elemental abundances in Mars-analog hypersaline lakes ##

Created by: Floyd Nichols

Email: floydnichols@vt.edu

# 1. Installing libraries that are not pre-installed on Google Colab

Googel Colab provides many pre-installed libraries; however, some libraries may need to be installed on your device. Uncomment the following libraries if a library needs to be installed.

In [ ]:
%%capture
# Installing Libraries

!pip install imblearn
!pip install statsmodels

# 2. Load Necessary Libraries and Dependencies

In [ ]:
# Housekeeping Libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from numpy import mean, std

# Libraries Necessary for Random Forest
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import imblearn
from imblearn.over_sampling import SMOTE

# Model Metric Libraries
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold, LearningCurveDisplay, ShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

# 3. Load Dataset

In this section we will load in our dataset. Since we are using a Google Colab notebook this can be done in two ways.
1. The data may be mounted to a google drive folder and loaded in or
2. The data may be uploaded to github and the url can be used.

In [ ]:
# Load the data using a github URL
url = ('https://raw.githubusercontent.com/FloydNichols97/OMAP/main/ML_Data_2.csv')
Data = pd.read_csv(url)
Data = Data.dropna()

Data.head() # Show the first few rows of the dataframe to check its structure and content

## 4. Data Preprocessing

## 4.1 Check for Linearity
Prior to the construction of our random forest model, we will first assess the linearity of our data.


In [ ]:
Data2 = Data.drop(columns = 'Sample')
Pearson = Data2.corr(method = 'pearson') # pearson is used to assess the linear relationship between variables
Spearman = Data2.corr(method = 'spearman') # spearman is used to assess the monotonic relationship between variables

# Visualize Correlations
plt.figure(figsize=(10,10))
sns.set_style("whitegrid")

# Pearson Correlation
plt.subplot(211)
sns.heatmap(Pearson, annot = False)
plt.title('Pearson Correlation Matrix')

# Spearman Correlation
plt.subplot(212)
sns.heatmap(Spearman, annot = False)
plt.title('Spearman Correlation Matrix')

Based on our correlation calculations, it is very likely that are data is highly complex with a mix of linear and non-linear relationships between the elemental variables. This suggests that machine learning approaches that are better equipped to handle such data should be employed. In the following sections, we will see if this is true.

## 4.2 Data Manipulation

Now, let's load in our matrix of explanatory variables (X) and our response variables (y). Since classifier algorithms require discrete categories we will encode boundary conditions to classify the total organic carbon values.

In [ ]:
# Define our boundary conditions

Data['Productivity'] = np.where(Data['TOC'] <= 2.5, 'low',
                                np.where(Data['TOC'] <= 10, 'moderate', 'high'))

X = Data.drop(columns=['Sample', 'TOC', 'Productivity']) # Add your categorical variable column(s) within the quotations to separate out non-numerical values
y = Data['Productivity'] # Add your categorical variable column within the quotations to create a y variable of your categories

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25) # This function splits both X and y variables in training a test splits. Specifically we will save 25% of the data for testing and use 75% for training

# 5. Random Forest (Classification)
As previously mentioned, some supervised learning algorithms excel at different data types (i.e. linear or non-linear relationships). For instance, Random Forest is exceptional at handling and predicting highly complex and non-linear data. Conversely, algorithms such as Support Vector Machine excel at predicting linear data.

## 5.1 Model Construction (Random Forest Classifier)

In [ ]:
# Setting up the model
RF_class = RandomForestClassifier(n_estimators = 10, max_depth = 5, min_samples_split = 2, bootstrap=True, random_state=42) # Instantiate Random Forest Model
RF_class.fit(X_train, y_train) # fit your model parameters to your training split
RF_predictions = RF_class.predict(X_test) # make a prediction of your test split based on the model defined above
RF_cv = RepeatedStratifiedKFold(n_splits=20, n_repeats=10, random_state=1) # account for overfitting by implementing a repeated stratified k fold cross-validation
RF_scores = cross_val_score(RF_class, X, y, scoring='accuracy', cv=RF_cv, n_jobs=-1, error_score='raise') # calculate cross-validation scores
print('Accuracy: %.3f (%.3f)' % ((mean(RF_scores), std(RF_scores)))) # print the accuracy of the model

## 5.2 Model Post-Processing (Compute a Confusion Matrix)
This step is not necessary; however, it is extremely valuable for examining your model. If you want to visualize your model to understand how well it is able to correctly label a class, you can compute a confusion matrix that plots the predicted label against the true label. Values along the diagonal represent the percentage of correctly labeled classes (true positive) whereas values outside the diagonal represents the percentage of misclassified classes.

In [ ]:
print(classification_report(y_test, RF_predictions)) # To get a wholistic view of the model we will print a classification report to examine the precision, recall, F1 score, and accuracy of the individual classes

cm = confusion_matrix(y_test, RF_predictions, labels=RF_class.classes_) # define confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=RF_class.classes_)
disp = disp.plot(cmap=plt.cm.Greens,values_format='g') # create a display plot for your confusion matrix
plt.title('Random Forest', fontsize = 16) # add a title with a font size of 16
plt.grid(False) # remove grid lines

From computing the classification report, we can more clearly see that although our model has a decent accuracy of 83%, it does a very poor job with classifying the 'High' class as indicated by the precision, recall, and F1 scores. Additionally, we see in the confusion matrix that there is a high rate of false negatives as well as a very large imbalance for the 'High' class compared to 'Low' and 'Moderate' classes.

So what does this mean for our model? Using these metrics it becomes clear that we first need to address the problem of sample size and imbalance for the 'High' class.

## 5.3 Feature Extraction
**Mean Decrease in Impurity**

Many models such as random forest can be considered black-box models due to their lack of explainability and interpretability. However, an advantage to using random forest is that post-hoc methods are available such as Mean Decrease in Impurity (MDI) or permutation importance which can be used to provide explainability to the model.

In [ ]:
# Compute Feature Importance by Mean Decrease in Impurity (MDI)
importances = RF_class.feature_importances_
sorted_indices = np.argsort(importances)[::-1]
feat_labels = Data.columns[1:]

# Create Bar Chart for Feature Importances
plt.figure(figsize=(12,5))
sns.set_style("whitegrid")

plt.title('Feature Importance (MDI)', fontsize = 24)
plt.bar(range(X_train.shape[1]), importances[sorted_indices], align='center', color = '#D3D3D3', edgecolor = 'k', linewidth = 2)
plt.xticks(range(X_train.shape[1]), X_train.columns[sorted_indices], rotation=90)
plt.tight_layout()
plt.tick_params(labelsize=24)

plt.savefig("Feature_Importance.png", format="png", bbox_inches="tight", transparent=True)

## 5.4 Assess Generalizability of Model (Learning Curve)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))

common_params = {
    "X": X,
    "y": y,
    "train_sizes": np.linspace(0.1, 1.0, 10),
    "cv": ShuffleSplit(n_splits=20, test_size=0.2, random_state=42),
    "score_type": "both",
    "n_jobs": 4,
    "line_kw": {"marker": "o"},
    "std_display_style": "fill_between",
    "score_name": "Accuracy",
}

LearningCurveDisplay.from_estimator(RF_class, **common_params, ax=ax)
handles, label = ax.get_legend_handles_labels()
ax.legend(handles[:2], ["Training Score", "Test Score"])
ax.set_title(f"Learning Curve for {RF_class.__class__.__name__}")

# 6. Random Forest (Regression)

## 6.1 Model Construction (Random Forest Regression)

In [ ]:
X = Data.drop(columns=['Sample', 'TOC', 'Productivity']) # Add your categorical variable column(s) within the quotations to separate out non-numerical values
y = Data['TOC'] # Add your response variable column within the quotations to create a y variable of your categories

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25) # This function splits both X and y variables in training a test splits. Specifically we will save 25% of the data for testing and use 75% for training

In [ ]:
# Random Forest Regression
RF_reg = RandomForestRegressor(n_estimators = 10, max_depth = 5, min_samples_split = 2, bootstrap = True, random_state = 42) # Instantiate and define model parameters
RF_reg.fit(X_train, y_train) # fit your model parameters to your training split
RF_predictions = RF_reg.predict(X_test) # make a prediction of your test split based on the model defined above

## 6.2 Model Post-Processing (Compute Loss Functions)
Unlike classifier algorithms, an accuracy is not returned for regression problems. Instead, to understand how well a model is performing for a regression algorithm other means are computed including mean squared error (MSE), root mean squared error (rmse), mean absolute error (MAE), and/or mean absolute percentage error (MAPE).

In [ ]:
print("Root Mean squared error: %.2f" % np.sqrt(mean_squared_error(y_test, RF_predictions))) # compute and print root mean squared error
print("Coefficient of determination: %.2f" % r2_score(y_test, RF_predictions)) # compute and print the coefficient of determination: 1 is perfect prediction
print("Mean Absolute Error: %.2f" % mean_absolute_error(y_test, RF_predictions)) # compute and print mean absolute error
print("Mean Absolute Percent Error: %.2f" % mean_absolute_percentage_error(y_test, RF_predictions)) # compute and print mean absolute percentage error

## 6.3 Visualize Model Peformance

In [ ]:
plt.axline((0, 0), slope=1, color='k', ls='--', label = 'Observed (1:1 Correlation)')
sns.scatterplot(x = y_test, y = RF_predictions, edgecolor = 'k', alpha = 0.6, label = f'Random Forest: RMSE = {np.sqrt(mean_squared_error(y_test, RF_predictions)).round(2)}')
plt.xlabel("TOC% (Observed)")
plt.ylabel("TOC% (Predicted)")

## 6.4 Assess Generalizability of Model (Learning Curve)

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4, 4))

common_params = {
    "X": X,
    "y": y,
    "train_sizes": np.linspace(0.1, 1.0, 10),
    "cv": ShuffleSplit(n_splits=20, test_size=0.2, random_state=42),
    "score_type": "both",
    "n_jobs": 4,
    "line_kw": {"marker": "o"},
    "std_display_style": "fill_between",
    "score_name": "neg_root_mean_squared_log_error",
}

LearningCurveDisplay.from_estimator(RF_reg, **common_params, ax=ax)
handles, label = ax.get_legend_handles_labels()
ax.legend(handles[:2], ["Training Score", "Test Score"])
ax.set_title(f"Learning Curve for {RF_reg.__class__.__name__}")

# Extra Activity

#### Data Augmentation Using SMOTE for Imbalanced Datasets

In the above section, we see that our model showed a false sense of good performance due to an imbalance in classes, specifically that of the 'High' class. In the following section, we will employ data augmentation using a Synthetic Minority Oversampling Technique (SMOTE) to see if we can improve the overall performance of the model.

In [ ]:
# Define Data Augmentation Variables
oversample = SMOTE() # Create an object storing the SMOTE function
X, y = oversample.fit_resample(X, y) # Resample data using the Synthetic Minority Oversampling Technique
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25)

# Setting up the model
RF_model_aug = RandomForestClassifier(n_estimators = 30, min_samples_split = 6, bootstrap=True, random_state=42) # Instantiate Random Forest Model
RF_model_aug.fit(X_train, y_train) # fit your model parameters to your training split
RF_aug_predictions = RF_model_aug.predict(X_test) # make a prediction of your test split based on the model defined above
RF_aug_cv = RepeatedStratifiedKFold(n_splits=20, n_repeats=10, random_state=1) # account for overfitting by implementing a repeated stratified k fold cross-validation
RF_aug_scores = cross_val_score(RF_model_aug, X, y, scoring='accuracy', cv=RF_aug_cv, n_jobs=-1, error_score='raise') # calculate cross-validation scores
print('Accuracy: %.3f (%.3f)' % ((mean(RF_aug_scores), std(RF_aug_scores)))) # print the accuracy of the model

In [ ]:
print(classification_report(y_test, RF_aug_predictions)) # To get a wholistic view of the model we will print a classification report to examine the precision, recall, F1 score, and accuracy of the individual classes

cm = confusion_matrix(y_test, RF_aug_predictions, labels=RF_model_aug.classes_) # define confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=RF_model_aug.classes_)
disp = disp.plot(cmap=plt.cm.Greens,values_format='g') # create a display plot for your confusion matrix
plt.title('Data Augmented Random Forest', fontsize = 16) # add a title with a font size of 16
plt.grid(False) # remove grid lines

Voila! Using a data augmentation technique, we do not drastically improve the model accuracy; however, more importantly we do improve the overall generalization of the model. As such, we decrease the false negative rate of the 'High' class, ultimately, improving the individual class metrics for precision, recall, F1 score, and accuracy for our 'High' class.